# Pydantic AI search diagnostics

This notebook intentionally ignores the project prompts. It checks the real chain in three steps:

1. model returns a plain answer without tools;
2. model may call the real `search_spots` tool and returns plain text;
3. model must call the real `search_spots` tool and return a structured Pydantic object.


In [6]:
import json
import asyncio
import os
from pathlib import Path
from pprint import pprint

from pydantic import BaseModel, Field
from pydantic_ai import Agent

from capabilities.search import SearchCapability
from pydantic_ai.capabilities import Thinking
from core.models import StargazingSpot


async def run_with_timeout(label: str, awaitable, timeout: float = 45):
    print(f"START: {label}")
    result = await asyncio.wait_for(awaitable, timeout=timeout)
    print(f"DONE: {label}")
    return result


def load_dotenv(path: str = ".env") -> None:
    env_path = Path(path)
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ[key] = value


load_dotenv()

MODEL = os.environ["LAZY_STELLAR_MODEL"]

print(f"MODEL: {MODEL}")
for key in ["OPENROUTER_API_KEY", "OPENAI_API_KEY", "TAVILY_API_KEY"]:
    print(f"{key}: {'set' if os.getenv(key) else 'missing'}")


MODEL: openrouter:google/gemini-3.1-flash-lite
OPENROUTER_API_KEY: set
OPENAI_API_KEY: missing
TAVILY_API_KEY: set


In [7]:
class SpotSearchReport(BaseModel):
    spots_found: int = Field(description="Number of spots returned by the search tool.")
    summary: str = Field(description="Concise user-facing answer in English.")
    spots: list[StargazingSpot] = Field(default_factory=list)


## 1. Plain model call, no tools

If this hangs or fails, the problem is model/provider configuration, not search or structured output.


In [8]:
plain_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    instructions="Reply with one short sentence. Do not use tools.",
)

plain_result = await run_with_timeout(
    "plain model call",
    plain_agent.run("Say hello and name one reason dark skies matter."),
)

print("OUTPUT:")
print(plain_result.output)
print("\nUSAGE:")
print(plain_result.usage)


START: plain model call
DONE: plain model call
OUTPUT:
Hello, dark skies are essential for protecting the natural nocturnal behaviors of wildlife.

USAGE:
RunUsage(input_tokens=21, output_tokens=15, details={'is_byok': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'image_tokens': 0}, requests=1)


## 2. Real search tool, plain text output

If step 1 works but this fails, inspect whether the model called `search_spots`, whether Tavily/DuckDuckGo failed, or whether the tool result came back as an error string.


In [9]:
search_text_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    capabilities=[SearchCapability()],
    instructions=(
        "You are a stargazing assistant. For location-specific recommendations, "
        "call search_spots exactly once before answering. Then summarize the returned spots. "
        "If the tool returns an error string, report that exact failure briefly."
    ),
)

search_text_result = await run_with_timeout(
    "search tool + plain text output",
    search_text_agent.run(
        "Find 3 stargazing spots near Paris, reachable without a private car."
    ),
    timeout=90,
)

print("OUTPUT:")
print(search_text_result.output)
print("\nUSAGE:")
print(search_text_result.usage)
print("\nMESSAGES:")
for message in search_text_result.all_messages():
    print(type(message).__name__, message)


START: search tool + plain text output


/Users/johnwunderbellen/my-lazy-stellar/capabilities/search.py:27: LogfireNotConfiguredWarning: No logs or spans will be created until `logfire.configure()` has been called. Set the environment variable LOGFIRE_IGNORE_NO_CONFIG=1 or add ignore_no_config=true in pyproject.toml to suppress this warning.
  logfire.info("Using Tavily Search API for active query: {query}", query=full_query)


DONE: search tool + plain text output
OUTPUT:
Finding dark skies near a major metropolitan area like Paris can be challenging, as light pollution significantly affects visibility. While the search results highlight major Dark Sky Reserves (such as the Morvan or Pic du Midi), those are distant locations across France.

For stargazing accessible from Paris without a private car, you are generally limited to regional parks reachable by regional rail (Transilien or TER) that are far enough from the city center to reduce glare. Here are three recommended areas:

1.  **Parc Naturel Régional de la Haute Vallée de Chevreuse:**
    *   **How to get there:** Take the RER B to Saint-Rémy-lès-Chevreuse. From here, you can hike or use local bus connections toward more rural areas like Dampierre or Senlisse.
    *   **Why it's good:** It is protected from some urban development and offers pockets of darkness once you move away from the town centers.

2.  **Parc Naturel Régional du Vexin Français:**


## 2a. Real search tool, JSON text output

This tests whether the model can use the search tool and emit JSON text when Pydantic AI does not force the final `output_type` tool.


In [10]:
from pydantic_ai.capabilities import WebSearch
from pydantic_ai.capabilities import Thinking


json_text_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    # capabilities=[SearchCapability()],
    capabilities=[WebSearch(), Thinking("high")],
    instructions=(
        "You are a stargazing assistant. You must call search_spots exactly once before final output. "
        "Then return ONLY valid JSON text with keys: spots_found, summary, spots. "
        "spots must be an array of objects with name, latitude, longitude, source, description, accessibility, safety_assessment, and bortle_class. "
        "If search_spots returns an error string, return {\"spots_found\": 0, \"summary\": <error>, \"spots\": []}. "
        "Do not wrap the JSON in markdown. Do not invent spots that were not returned by the tool."
    ),
)

json_text_result = await run_with_timeout(
    "search tool + JSON text output",
    json_text_agent.run(
        "Find 3 stargazing spots near Paris, reachable without a private car."
    ),
    timeout=90,
)

print("RAW OUTPUT:")
print(json_text_result.output)
print("\nPARSED JSON:")
pprint(json.loads(json_text_result.output))
print("\nUSAGE:")
print(json_text_result.usage)
print("\nMESSAGES:")
for message in json_text_result.all_messages():
    print(type(message).__name__, message)


/var/folders/lt/75xqxt7561731l87vw1h2_3w0000gn/T/ipykernel_1366/4026942807.py:9: PydanticAIDeprecationWarning: WebSearch will stop auto-selecting DuckDuckGo based on package availability in v2. To keep this fallback, pass `local='duckduckgo'` (or `local=True`). To disable the fallback, pass `local=False`.
  capabilities=[WebSearch(), Thinking("high")],


START: search tool + JSON text output
DONE: search tool + JSON text output
RAW OUTPUT:
{
"spots_found": 3,
"summary": "Three stargazing spots near Paris were identified that are accessible via the regional train network (Transilien/TER). These locations offer varying degrees of distance from Paris's light pollution, with forests like Fontainebleau and Rambouillet providing the best opportunities for darker skies compared to the immediate suburbs.",
"spots": [
{
"name": "Fontainebleau Forest",
"latitude": 48.4239,
"longitude": 2.7011,
"source": "https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGtKvhnnx4QT5woxuIzLsgKQH_wz9rNjdtGO6rZGnFpWp99lUVqLpNSgfyUZZBNtosB2GOEbHJbzuBxbLVkjD8W5WEKFbVUad0KHzP5SFmYmYEx6IP_HcBBUjqbhc0pT4WhihYK0Q-gFjS5on-i6dMdZU6sfs7pu3-rgPyrJAfGovscKdKwog==",
"description": "A vast forest area known for its rocky landscapes and royal hunting history, located roughly 60km from Paris, offering significantly better dark skies than the capital.",
"accessi

## 3. Real search tool, structured Pydantic output

If steps 1 and 2 work but this fails, the problem is structured output/tool interaction. This cell forces a `SpotSearchReport` final result.


In [ ]:
structured_agent = Agent(
    model=MODEL,
    defer_model_check=True,
    output_type=SpotSearchReport,
    capabilities=[WebSearch(), Thinking(effort=True)],
    instructions=(
        "You are a stargazing assistant. You must use WebSearch(). "
        "Use the tool result to fill SpotSearchReport. "
        "If search_spots returns an error string, return spots_found=0, spots=[], and put the error in summary. "
        "Do not invent spots that were not returned by the tool."
        "before returning rechack all fields for correctness and consistency, and if you find any issues, use the search tool again"
        "You can use search tool max 2 times for each spot"
        "Add the UTC offset for each spot based on location"
    ),
)

structured_result = await run_with_timeout(
    "search tool + structured Pydantic output",
    structured_agent.run(
        "Find 3 stargazing spots near Paris, reachable with a private car, and in range of 1 hour or 80 km."
    ),
    timeout=90,
)

print("PYDANTIC OUTPUT:")
pprint(structured_result.output.model_dump())
print("\nJSON:")
print(structured_result.output.model_dump_json(indent=2))
print("\nUSAGE:")
print(structured_result.usage)
print("\nMESSAGES:")
for message in structured_result.all_messages():
    print(type(message).__name__, message)


/var/folders/lt/75xqxt7561731l87vw1h2_3w0000gn/T/ipykernel_1366/4059372365.py:5: PydanticAIDeprecationWarning: WebSearch will stop auto-selecting DuckDuckGo based on package availability in v2. To keep this fallback, pass `local='duckduckgo'` (or `local=True`). To disable the fallback, pass `local=False`.
  capabilities=[WebSearch(), Thinking(effort=True)],


START: search tool + structured Pydantic output
DONE: search tool + structured Pydantic output
PYDANTIC OUTPUT:
{'spots': [{'accessibility': 'Accessible by car from Paris (approx. 45-60 '
                             'minutes). Public parking available near historic '
                             'sites like Château de la Madeleine.',
            'additional_info': None,
            'bortle_class': 5,
            'description': 'A protected natural area southwest of Paris known '
                           'for its rolling landscapes and forests. Several '
                           'municipalities within the park have received the '
                           '"Village Étoilé" label for their efforts to reduce '
                           'light pollution, providing a relatively dark sky '
                           'compared to the city center.',
            'latitude': 48.667,
            'longitude': 2.05,
            'name': 'Parc Naturel Régional de la Haute Vallée de Chevreuse',

: 

In [ ]:
raw = [0,0,0,0,0,0,0]
gameboard = [raw,raw,raw,raw,raw,raw]

In [ ]:
print(raw)

3


In [ ]:
n=10 
for n in range(number-1):
    print(n)